# 第21講：次元削減と教師なし学習

第20講までは「数学の点数」「週給」「ペンギンの品種」のように**正解ラベル $y$ があるデータ**（教師あり学習）を扱ってきた。
しかし現実の経済学・経営学では、「どの顧客がVIP客か」「無数の経済指標をどう1枚の地図にまとめるか」のように、**最初から正解ラベル $y$ が存在しない課題**が数多くある。

このノートブックでは、**教師なし学習**の2大手法である**主成分分析**（PCA）と**k平均法**（K-Means）を**4つの実践タスク**で学び、各タスクの直後に**実習①〜実習④**を行う。

1. **タスク①＋実習①**：**主成分分析**（PCA）の基本と総合指標（高校データ `mhigh.csv`）
2. **タスク②＋実習②**：世界158か国の経済・社会マップを描く（世界幸福度データ `happy.csv`）
3. **タスク③＋実習③**：**k平均法**による顧客セグメンテーション（EC顧客データ `customers.csv`）
4. **タスク④＋実習④**：PCA × k平均法による47都道府県の商圏分析（都道府県データ `pref47.csv`）

## 0. 準備

最初に1回だけ実行する。グラフの軸に日本語を出すため、Noto Sans JP を登録する。

In [ ]:
import piplite
import pyodide
await piplite.install("seaborn")

import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
from pyodide.http import pyfetch, open_url
%matplotlib inline

resp = await pyfetch(
    "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansjp/NotoSansJP%5Bwght%5D.ttf"
)
with open("NotoSansJP.ttf", "wb") as f:
    f.write(await resp.bytes())
fm.fontManager.addfont("NotoSansJP.ttf")

sns.set_theme(style="darkgrid", font="Noto Sans JP")

---
## 1. タスク①：主成分分析（PCA）の基本（高校データ `mhigh.csv`）

**主成分分析**（PCA: Principal Component Analysis）は、多数の数値列を**情報のロスが最小になるように少数の総合スコアへ圧縮（次元削減）する**手法である。

### 1-1. なぜPCAの前に必ず「標準化」が必要なのか？

PCAは「数値のばらつき（分散）が大きい方向＝重要な情報」と判断して軸を引く。
単位や桁数が大きく異なる都道府県データ（`pref47.csv`）で、**標準化しない場合**と**標準化した場合**の違いを確かめてみよう。

In [ ]:
from sklearn.decomposition import PCA

url_pf = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/pref47.csv"
pref_demo = pd.read_csv(open_url(url_pf))
demo_cols = ["population_man", "area_km2", "life_expectancy",
             "ramen_per_100k", "starbucks_count", "avg_income_man"]

# 【NG】標準化せずにそのままPCAをかけた場合（面積の桁が大きすぎて99.9%支配してしまう）
pca_ng = PCA(n_components=2).fit(pref_demo[demo_cols])
print("【NG：標準化なし】PC1の寄与率:", pca_ng.explained_variance_ratio_[0].round(4))

# 【OK】平均0・標準偏差1に標準化してからPCAをかけた場合
X_demo_std = (pref_demo[demo_cols] - pref_demo[demo_cols].mean()) / pref_demo[demo_cols].std()
pca_ok = PCA(n_components=2).fit(X_demo_std)
print("【OK：標準化あり】PC1の寄与率:", pca_ok.explained_variance_ratio_[0].round(4))

### 1-2. 高校データ（677人）の4変数を2つの主成分に圧縮する

高校データ（`mhigh.csv`）の4つの数値列（国語 `japanese`・数学 `math`・身長 `height`・体重 `weight`）を標準化し、`PCA(n_components=2)` で2つの総合指標（`PC1`, `PC2`）に要約しよう。

In [ ]:
url_mh = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/mhigh.csv"
mhigh = pd.read_csv(open_url(url_mh))
mhigh.columns = ["name", "katakana", "sex", "grade",
                 "japanese", "math", "total", "height", "weight", "class"]

cols = ["japanese", "math", "height", "weight"]
X_std = (mhigh[cols] - mhigh[cols].mean()) / mhigh[cols].std()

pca = PCA(n_components=2)
scores = pca.fit_transform(X_std)

print("各主成分の寄与率:", pca.explained_variance_ratio_.round(3))
print("2軸の累積寄与率:", pca.explained_variance_ratio_.sum().round(3))

### 1-3. 主成分負荷量（重み）から軸の正体を読み解く

`pca.components_` には、元の4変数が `PC1` と `PC2` にどれだけ効いているかを表す重み（**主成分負荷量**）が入っている。
表にして確認し、生徒677人の2次元散布図を描いてみよう。

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=cols, columns=["PC1", "PC2"])
print(loadings.round(2))

mhigh["PC1"] = scores[:, 0]
mhigh["PC2"] = scores[:, 1]

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=mhigh, x="PC1", y="PC2", hue="class",
    hue_order=sorted(mhigh["class"].unique()), palette="viridis", s=35, alpha=0.8
)
plt.xlabel("PC1：体格スコア（身長・体重がプラス）")
plt.ylabel("PC2：学力スコア（国語・数学がプラス）")
plt.title("高校677人の主成分マップ（4列の87.0%を保持）")
plt.legend(title="クラス", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 実習①：高校データの主成分分析（PCA）

次の4つを下のコードセルで実行しなさい。
1. `mhigh.csv` を読み込み、列名を `name`, `katakana`, `sex`, `grade`, `japanese`, `math`, `total`, `height`, `weight`, `class` に変更する
2. 4つの数値列 `cols = ["japanese", "math", "height", "weight"]` を標準化した表 **`X_mh_std`** を作る
3. `PCA(n_components=2)` を `X_mh_std` に適用し、2つの主成分の**寄与率**（`.explained_variance_ratio_.round(3)`）と**累積寄与率**（`.sum().round(3)`）を表示する
4. 主成分スコアを `mhigh["PC1"]`, `mhigh["PC2"]` に保存し、クラス別（`groupby("class")`）の `PC1` と `PC2` の平均値を `.round(2)` で表示する

In [ ]:
# 実習①：ここにコードを書いて実行する
mhigh = pd.read_csv(open_url(url_mh))
mhigh.columns = ["name", "katakana", "sex", "grade",
                 "japanese", "math", "total", "height", "weight", "class"]

cols = ["japanese", "math", "height", "weight"]
X_mh_std = (mhigh[cols] - mhigh[cols].mean()) / mhigh[cols].std()

pca_mh = PCA(n_components=2)
scores_mh = pca_mh.fit_transform(X_mh_std)
print("寄与率:", pca_mh.explained_variance_ratio_.round(3))
print("累積寄与率:", pca_mh.explained_variance_ratio_.sum().round(3))

mhigh["PC1"] = scores_mh[:, 0]
mhigh["PC2"] = scores_mh[:, 1]
print(mhigh.groupby("class")[["PC1", "PC2"]].mean().round(2))

---
## 2. タスク②：世界158か国の経済・社会マップを描く（世界幸福度データ `happy.csv`）

国連の世界幸福度報告データ（`happy.csv`、158か国）には、1人あたりGDP・社会的支援・健康寿命・選択の自由度・政府への信頼・寛容さ（寄付文化）の**6つの経済・社会指標**が含まれている。
答えにあたる `Happiness Score` は一切使わず、**6指標だけ**から世界の構造を2次元マップに要約してみよう。

In [ ]:
url_hap = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/happy.csv"
happy = pd.read_csv(open_url(url_hap))
happy.head(3)

### 2-1. 6指標すべての寄与率・累積寄与率と主成分負荷量を確認する

`PCA()` のカッコ内を空欄にして実行すると、6個すべての主成分の寄与率がわかる。
上位2軸（`PC1`, `PC2`）だけで全体の **70.0%** の情報量を保持できていることを確認しよう。

In [ ]:
hap_cols = [
    "Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
    "Freedom", "Trust (Government Corruption)", "Generosity"
]
X_hap = (happy[hap_cols] - happy[hap_cols].mean()) / happy[hap_cols].std()

pca_all = PCA().fit(X_hap)
print("各主成分の寄与率:", pca_all.explained_variance_ratio_.round(3))
print("累積寄与率:", np.cumsum(pca_all.explained_variance_ratio_).round(3))

loadings_hap = pd.DataFrame(pca_all.components_[:2].T, index=hap_cols, columns=["PC1", "PC2"])
print("\n--- 上位2軸の主成分負荷量 ---")
print(loadings_hap.round(2))

### 2-2. 世界158か国の2次元マップを描き、日本の立ち位置を確認する

`PC1`（経済・生活基盤の豊かさ）を横軸、`PC2`（寛容さ・信頼・自由度）を縦軸にとり、世界158か国の散布図上で**日本（Japan）**がどこに位置するか可視化する。

In [ ]:
scores_hap = PCA(n_components=2).fit_transform(X_hap)
happy["PC1"] = scores_hap[:, 0]
happy["PC2"] = scores_hap[:, 1]

plt.figure(figsize=(8, 5))
sns.scatterplot(data=happy, x="PC1", y="PC2", s=40, alpha=0.7, color="steelblue", label="各国（158か国）")

jp = happy[happy["Country"] == "Japan"].iloc[0]
plt.scatter(jp["PC1"], jp["PC2"], color="crimson", s=160, marker="*", zorder=5, label="Japan（日本）")
plt.annotate(
    f"Japan (PC1={jp['PC1']:.2f}, PC2={jp['PC2']:.2f})",
    xy=(jp["PC1"], jp["PC2"]), xytext=(jp["PC1"] - 1.6, jp["PC2"] - 0.6),
    color="crimson", fontweight="bold", arrowprops=dict(arrowstyle="->", color="crimson")
)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("PC1：経済・生活基盤の豊かさ（47.9%）")
plt.ylabel("PC2：寛容さ・信頼・自由度（22.1%）")
plt.title("PCAで描いた世界158か国の2次元経済・社会マップ")
plt.legend()
plt.tight_layout()
plt.show()

### 実習②：世界158か国の経済・社会マップ分析

次の4つを下のコードセルで実行しなさい。
1. `happy.csv` を読み込み、6つの指標列（`Economy (GDP per Capita)`, `Family`, `Health (Life Expectancy)`, `Freedom`, `Trust (Government Corruption)`, `Generosity`）を標準化した表 **`X_hap`** を作る
2. `PCA(n_components=2)` を `X_hap` に適用し、スコアを `happy["PC1"]`, `happy["PC2"]` に保存して2軸の**寄与率**と**累積寄与率**を表示する
3. `happy[happy["Country"] == "Japan"][["Country", "Happiness Rank", "PC1", "PC2"]]` を表示し、日本のスコアを確認する
4. `happy["PC1"].corr(happy["Happiness Score"]).round(3)` を計算し、`PC1` と幸福度スコアの相関係数を確認する

In [ ]:
# 実習②：ここにコードを書いて実行する
happy = pd.read_csv(open_url(url_hap))
hap_cols = [
    "Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
    "Freedom", "Trust (Government Corruption)", "Generosity"
]
X_hap = (happy[hap_cols] - happy[hap_cols].mean()) / happy[hap_cols].std()

pca_hap2 = PCA(n_components=2)
scores_hap2 = pca_hap2.fit_transform(X_hap)
happy["PC1"] = scores_hap2[:, 0]
happy["PC2"] = scores_hap2[:, 1]

print("寄与率:", pca_hap2.explained_variance_ratio_.round(3))
print("累積寄与率:", pca_hap2.explained_variance_ratio_.sum().round(3))
print(happy[happy["Country"] == "Japan"][["Country", "Happiness Rank", "PC1", "PC2"]].round(2))
print("PC1と幸福度スコアの相関係数:", happy["PC1"].corr(happy["Happiness Score"]).round(3))

---
## 3. タスク③：k平均法による顧客セグメンテーション（EC顧客データ `customers.csv`）

マーケティングでは、会員全員に同じ割引クーポンを配るのではなく、行動データの似た顧客をグループ（**クラスター／セグメント**）に分けて施策を打ち分ける。
**k平均法**（K-Means）を使って、ECサイト会員240人（`customers.csv`）を自動分類してみよう。

In [ ]:
from sklearn.cluster import KMeans

url_cu = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/customers.csv"
customers = pd.read_csv(open_url(url_cu))
customers.head(5)

### 3-1. エルボー法で最適なグループ数 $k$ を探す

k平均法も「距離」を使って近さを測るため、最初に必ず4列（年齢 `age`・年収 `annual_income_man`・月訪問回数 `visit_monthly`・年間購入額 `spending_man`）を**標準化**する。
その後、$k=1 \sim 8$ の**クラスター内誤差平方和**（SSE: `inertia_`）を計算して折れ線グラフを描き、「ひじ（Elbow）」のように折れ曲がる $k$ を確認しよう。

In [ ]:
c_cols = ["age", "annual_income_man", "visit_monthly", "spending_man"]
X_cust = (customers[c_cols] - customers[c_cols].mean()) / customers[c_cols].std()

sse_list = []
for k in range(1, 9):
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_cust)
    sse_list.append(km_temp.inertia_)

plt.figure(figsize=(6.5, 4))
plt.plot(range(1, 9), sse_list, marker="o", linewidth=2)
plt.xlabel("クラスター数 k（グループの数）")
plt.ylabel("クラスター内誤差平方和 SSE (inertia_)")
plt.title("エルボー法：k=4（SSE=99.7）で大きく下げ止まる")
plt.tight_layout()
plt.show()

### 3-2. $k=4$ で顧客を分類し、セグメント別の特徴を可視化する

`KMeans(n_clusters=4, random_state=42, n_init=10)` で顧客240人を4つのセグメントに分類し、「年収 × 年間購入額」の散布図とグループ別の平均値を確認しよう。

In [ ]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
customers["cluster"] = km.fit_predict(X_cust)

print(customers.groupby("cluster")[c_cols].mean().round(1))

plt.figure(figsize=(7, 4.5))
sns.scatterplot(
    data=customers, x="annual_income_man", y="spending_man",
    hue="cluster", palette="Set1", s=55, alpha=0.85
)
plt.xlabel("年収（万円）")
plt.ylabel("年間購入金額（万円）")
plt.title("発見された4つの顧客セグメント（Cluster 3＝高所得・未開拓層）")
plt.tight_layout()
plt.show()

### 実習③：EC顧客データのエルボー法とセグメンテーション

次の4つを下のコードセルで実行しなさい。
1. `customers.csv` を読み込み、4つの数値列 `c_cols = ["age", "annual_income_man", "visit_monthly", "spending_man"]` を標準化した表 **`X_cust`** を作る
2. `for k in range(1, 9):` ループで `KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_cust).inertia_` を計算して表示し、`k=4`（SSE = `99.7`）で下げ止まることを確認する
3. `KMeans(n_clusters=4, random_state=42, n_init=10)` の予測結果を `customers["cluster"]` に保存し、各クラスターの人数（`.value_counts()`）を表示する
4. `customers.groupby("cluster")[c_cols].mean().round(1)` を表示し、4つの顧客層の特徴を確認する

In [ ]:
# 実習③：ここにコードを書いて実行する
customers = pd.read_csv(open_url(url_cu))
c_cols = ["age", "annual_income_man", "visit_monthly", "spending_man"]
X_cust = (customers[c_cols] - customers[c_cols].mean()) / customers[c_cols].std()

for k in range(1, 9):
    sse = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_cust).inertia_
    print(f"k={k}: SSE = {sse:.1f}")

km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
customers["cluster"] = km4.fit_predict(X_cust)
print("\n--- 各クラスターの人数 ---")
print(customers["cluster"].value_counts().sort_index())
print("\n--- 各クラスターの平均値 ---")
print(customers.groupby("cluster")[c_cols].mean().round(1))

---
## 4. タスク④：PCA × k平均法による47都道府県の商圏分析（都道府県データ `pref47.csv`）

6列以上の地域データや店舗データを分析するときは、**「① 前処理（対数変換＋標準化） $\to$ ② k平均法でグループ分け $\to$ ③ PCAで2次元マップにして色分け可視化」**という連携が実務の王道である。
日本の47都道府県データ（`pref47.csv`）を4つの商圏タイプに分類し、**群馬県**がどの商圏に属するか調べてみよう。

In [ ]:
pref = pd.read_csv(open_url(url_pf))

# 1. 東京や北海道の極端な桁違いを np.log() で対数変換してから6列を標準化する
pref["log_pop"] = np.log(pref["population_man"])
pref["log_area"] = np.log(pref["area_km2"])
pref["log_sbux"] = np.log(pref["starbucks_count"])

p_cols = ["log_pop", "log_area", "life_expectancy",
          "ramen_per_100k", "log_sbux", "avg_income_man"]
X_pref = (pref[p_cols] - pref[p_cols].mean()) / pref[p_cols].std()

# 2. k平均法(k=4)で商圏グループを作り、PCA(2軸)で2次元マップの座標を作る
pref["cluster"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_pref)
pca_pref = PCA(n_components=2)
pref[["PC1", "PC2"]] = pca_pref.fit_transform(X_pref)

print("PCA 2軸の累積寄与率:", pca_pref.explained_variance_ratio_.sum().round(3))

### 4-1. 47都道府県の2次元商圏マップを可視化する

k平均法で分類した4つの商圏クラスターを色分けし、関東学園大学がある**群馬県**の位置に★マークを付けて確認しよう。

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=pref, x="PC1", y="PC2", hue="cluster",
    palette="tab10", s=70, alpha=0.85
)
gm = pref[pref["pref"] == "群馬県"].iloc[0]
plt.scatter(gm["PC1"], gm["PC2"], color="darkorange", edgecolor="black", s=200, marker="*", zorder=6)
plt.annotate(
    f"★群馬県 (Cluster {gm['cluster']})",
    xy=(gm["PC1"], gm["PC2"]), xytext=(gm["PC1"] - 2.0, gm["PC2"] + 0.5),
    fontweight="bold", arrowprops=dict(arrowstyle="->", color="black")
)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("PC1：都市規模・平均所得・スタバ店舗数（46.2%）")
plt.ylabel("PC2：面積・ラーメン外食文化（27.6%）")
plt.title("PCA × k平均法による日本47都道府県の2次元商圏マップ（累積寄与率 73.8%）")
plt.tight_layout()
plt.show()

### 実習④：47都道府県の商圏クラスタリングとPCA可視化

次の4つを下のコードセルで実行しなさい。
1. `pref47.csv` の `population_man`, `area_km2`, `starbucks_count` を対数変換し、残り3列（寿命・ラーメン・所得）と合わせた6列を標準化した表 **`X_pref`** を作る
2. `KMeans(n_clusters=4, random_state=42, n_init=10)` で4グループに分け、`pref["cluster"]` に保存して各クラスターの平均値（人口・所得・スタバ数・ラーメン店数）を表示する
3. `PCA(n_components=2)` のスコアを `pref["PC1"]`, `pref["PC2"]` に保存し、2軸の**累積寄与率**（`0.738`）を確認する
4. `pref[pref["pref"] == "群馬県"][["pref", "cluster", "PC1", "PC2"]]` と、群馬県と同じクラスターの道県一覧を表示する

In [ ]:
# 実習④：ここにコードを書いて実行する
pref = pd.read_csv(open_url(url_pf))
pref["log_pop"] = np.log(pref["population_man"])
pref["log_area"] = np.log(pref["area_km2"])
pref["log_sbux"] = np.log(pref["starbucks_count"])

p_cols = ["log_pop", "log_area", "life_expectancy",
          "ramen_per_100k", "log_sbux", "avg_income_man"]
X_pref = (pref[p_cols] - pref[p_cols].mean()) / pref[p_cols].std()

pref["cluster"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_pref)
summary_cols = ["population_man", "avg_income_man", "starbucks_count", "ramen_per_100k"]
print("--- 各商圏クラスターの平均値 ---")
print(pref.groupby("cluster")[summary_cols].mean().round(1))

pca_pf = PCA(n_components=2)
pref[["PC1", "PC2"]] = pca_pf.fit_transform(X_pref)
print("\n2軸の累積寄与率:", pca_pf.explained_variance_ratio_.sum().round(3))

print("\n--- 群馬県の分析結果 ---")
print(pref[pref["pref"] == "群馬県"][["pref", "cluster", "PC1", "PC2"]].round(2))

gunma_cl = pref.loc[pref["pref"] == "群馬県", "cluster"].iloc[0]
same_group = pref[pref["cluster"] == gunma_cl]["pref"].tolist()
print(f"群馬県と同じ Cluster {gunma_cl} の道県 ({len(same_group)}道県):", same_group)